# Solution 3B: Linear Models (Ridge and Lasso)
**BUSI 722: Data-Driven Finance II**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

df = pd.read_parquet("merged.parquet")
df = df[df["month"] >= "2023-01"].copy()
print(f"Loaded: {len(df):,} rows, {df['ticker'].nunique()} tickers")

FEATURES = ["momentum", "lag_month", "pb", "roe", "grossmargin",
            "assetturnover", "leverage", "asset_growth", "gp_to_assets"]

Loaded: 103,931 rows, 3866 tickers


In [2]:
# Rank features and target cross-sectionally each month
for feat in FEATURES:
    df[f"{feat}_rank"] = df.groupby("month")[feat].transform(lambda x: x.rank(pct=True))
df["ret_rank"] = df.groupby("month")["return"].transform(lambda x: x.rank(pct=True))
feat_rank_cols = [f"{f}_rank" for f in FEATURES]

# Train/test split
train = df[(df["month"] >= "2023-01") & (df["month"] <= "2024-12")].copy()
test = df[df["month"] >= "2025-01"].copy()
X_train = train[feat_rank_cols].values
y_train = train["ret_rank"].values
X_test = test[feat_rank_cols].values
print(f"Train: {len(train):,} rows ({train['month'].min()} to {train['month'].max()})")
print(f"Test:  {len(test):,} rows ({test['month'].min()} to {test['month'].max()})")

Train: 73,496 rows (2023-01 to 2024-12)
Test:  30,435 rows (2025-01 to 2025-11)


In [3]:
def compute_monthly_spearman(predictions, df_subset):
    df_sub = df_subset.copy()
    df_sub["pred"] = predictions
    result = {}
    for m, grp in df_sub.groupby("month"):
        if len(grp) > 10:
            rho, _ = spearmanr(grp["pred"], grp["return"])
            result[m] = rho
    return pd.Series(result).sort_index()

In [4]:
from sklearn.linear_model import Ridge, Lasso

## 1. Ridge regression

In [5]:
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
print("Ridge coefficients:")
for name, coef in zip(FEATURES, ridge.coef_):
    print(f"  {name:20s}: {coef:+.4f}")

Ridge coefficients:
  momentum            : +0.0568
  lag_month           : +0.0081
  pb                  : -0.0131
  roe                 : +0.0703
  grossmargin         : +0.0458
  assetturnover       : +0.0459
  leverage            : +0.0327
  asset_growth        : +0.0166
  gp_to_assets        : -0.0001


The largest positive coefficient indicates the most bullish signal; the largest negative coefficient indicates the most bearish.

## 2. Lasso regression

In [6]:
lasso = Lasso(alpha=0.001, max_iter=10000)
lasso.fit(X_train, y_train)
print("Lasso coefficients:")
for name, coef in zip(FEATURES, lasso.coef_):
    zeroed = " (zeroed)" if abs(coef) < 1e-6 else ""
    print(f"  {name:20s}: {coef:+.4f}{zeroed}")

Lasso coefficients:
  momentum            : +0.0456
  lag_month           : +0.0000 (zeroed)
  pb                  : +0.0000 (zeroed)
  roe                 : +0.0718
  grossmargin         : +0.0095
  assetturnover       : +0.0000 (zeroed)
  leverage            : +0.0213
  asset_growth        : +0.0078
  gp_to_assets        : +0.0123


Lasso sets some coefficients exactly to zero, performing automatic feature selection.

## 3-4. Spearman rank correlations on test set

In [7]:
ridge_pred = ridge.predict(X_test)
lasso_pred = lasso.predict(X_test)

sp_ridge = compute_monthly_spearman(ridge_pred, test)
sp_lasso = compute_monthly_spearman(lasso_pred, test)

print(f"Ridge  - Mean Spearman: {sp_ridge.mean():.4f}, Median: {sp_ridge.median():.4f}")
print(f"Lasso  - Mean Spearman: {sp_lasso.mean():.4f}, Median: {sp_lasso.median():.4f}")

Ridge  - Mean Spearman: 0.0521, Median: 0.0708
Lasso  - Mean Spearman: 0.0515, Median: 0.0514
